In [77]:
!pip install requests==2.32.5
!pip install -q langchain sentence-transformers faiss-cpu streamlit pypdf transformers

In [78]:
!pip install transformers==4.41.2
!pip install accelerate==0.30.1
!pip install sentence-transformers==2.7.0

In [79]:
import sys
!pip install -q langchain-community
!pip install langchain_text_splitters
from langchain_community.document_loaders import PyPDFLoader
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [80]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [81]:
# Load PDF file
loader=PyPDFLoader("/content/drive/MyDrive/Week-12.pdf")
# Extract documents (pages)
documents=loader.load()[1:]
# Print number of pages
print("Total pages:", len(documents))

Total pages: 70


In [82]:
# Print first page content
print(documents[0].page_content)

What is Data Analytics
 “Data analytics (DA) is the process of examining data sets in order to draw 
conclusions about the information they contain, increasingly with the aid of 
specialized systems and software. Data analytics technologies and techniques are 
widely used in commercial industries to enable organizations to make more‐
informed business decisions and by scien tists  and researchers to verify or disprove 
scientific models, theories and hypotheses.”
[An admin's guide to AWS data management]
2Introduction to Internet of Things
NPTEL


In [83]:
print(documents[0].page_content[:500])

What is Data Analytics
 “Data analytics (DA) is the process of examining data sets in order to draw 
conclusions about the information they contain, increasingly with the aid of 
specialized systems and software. Data analytics technologies and techniques are 
widely used in commercial industries to enable organizations to make more‐
informed business decisions and by scien tists  and researchers to verify or disprove 
scientific models, theories and hypotheses.”
[An admin's guide to AWS data m


In [84]:
#Text cleaning

def clean_text(text):
    text = str(text)
    # Replace newlines with period + space
    text = text.replace("\n", ". ")
    # Remove URLs
    text = re.sub(r"http\S+|www\S+", '', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    # Remove weird symbols like 
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    # fix broken dots
    text = re.sub(r'\s*\.\s*', '. ', text)
    # remove multiple dots
    text = re.sub(r'\.\.+', '.', text)
    # remove emails
    text = re.sub(r'\S+@\S+', '', text)
    return text

In [85]:
# Apply cleaning to all documents
for doc in documents:
    doc.page_content=clean_text(doc.page_content)
# Check result
print(documents[0].page_content[:500])

What is Data Analytics. Data analytics (DA) is the process of examining data sets in order to draw. conclusions about the information they contain, increasingly with the aid of. specialized systems and software. Data analytics technologies and techniques are. widely used in commercial industries to enable organizations to make more. informed business decisions and by scien tists and researchers to verify or disprove. scientific models, theories and hypotheses. . [An admin's guide to AWS data man


In [86]:
#Tect chunking

# Create Text Splitter
text_splitter= RecursiveCharacterTextSplitter(
    chunk_size=500,        # size of each chunk
    chunk_overlap=50      # overlap between chunks
)
# Split Documents into Chunks
texts= text_splitter.split_documents(documents)

In [87]:
# Check Results
print("Total chunks:", len(texts))
# View First Chunk
print(texts[0].page_content)

Total chunks: 79
What is Data Analytics. Data analytics (DA) is the process of examining data sets in order to draw. conclusions about the information they contain, increasingly with the aid of. specialized systems and software. Data analytics technologies and techniques are. widely used in commercial industries to enable organizations to make more. informed business decisions and by scien tists and researchers to verify or disprove. scientific models, theories and hypotheses. . [An admin's guide to AWS data


In [88]:
for i in range(3):
    print(f"\nChunk {i+1}:")
    print(texts[i].page_content[:200])


Chunk 1:
What is Data Analytics. Data analytics (DA) is the process of examining data sets in order to draw. conclusions about the information they contain, increasingly with the aid of. specialized systems an

Chunk 2:
and hypotheses. . [An admin's guide to AWS data management]. 2Introduction to Internet of Things. NPTEL

Chunk 3:
Types of Data Analysis. Two types of analysis. Qualitative Analysis. Deals with the analysis of data that is categorical in nature. Quantitative Analysis. Quantitative analysis refers to the process b


In [89]:
!pip install sentence-transformers faiss-cpu

In [90]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

In [91]:
# Embedding Model

# Create Embedding Model
embeddings= HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)
# Create FAISS Vector Store
vectorstore= FAISS.from_documents(texts, embeddings)

In [92]:
# Save FAISS Index
vectorstore.save_local("faiss_index")

In [93]:
# Load FAISS Later
vectorstore= FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

In [94]:
# Test Similarity Search
query= "What is data analytics?"
docs= vectorstore.similarity_search(query, k=3)
for i, doc in enumerate(docs):
    print(f"\nResult {i+1}:")
    print(doc.page_content[:200])


Result 1:
What is Data Analytics. Data analytics (DA) is the process of examining data sets in order to draw. conclusions about the information they contain, increasingly with the aid of. specialized systems an

Result 2:
Regression analysis. In statistical modeling, regression analysis is a statistical process for estimating the. relationships among variables. Focuses on the relationship between a dependent variable a

Result 3:
Regression analysis (Contd. ). The estimation target is a function of the independent variables called the. regression function. Characterize the variation of the dependent variable around the regress


In [95]:
# RAG SYSTEM
!pip install -U transformers==4.41.2

In [96]:
from transformers import pipeline
qa_pipeline = pipeline(
    "text2text-generation",
    model="google/flan-t5-base"
)

In [98]:
def ask_question(query):
    docs = vectorstore.similarity_search(query, k=4)

    context = " ".join([doc.page_content for doc in docs])[:800]

    prompt = f"""
Answer clearly and completely based on the context.
Answer ONLY based on the context.
If answer is not present, say "Answer Not found in document".

Context: {context}

Question: {query}

Answer:
"""

    response = qa_pipeline(prompt, max_new_tokens=120)

    # return response[0]['generated_text'].split("Answer:")[-1].strip()
    answer= response[0]['generated_text'].strip()
    answer = re.sub(r'\s*\.\s*', '. ', answer)
    answer = re.sub(r'\.\.+', '.', answer)
    return answer

In [99]:
import warnings
warnings.filterwarnings("ignore")

In [100]:
# Test It
print(ask_question("What is data analytics?"))

Data analytics (DA) is the process of examining data sets in order to draw. conclusions about the information they contain, increasingly with the aid of. specialized systems and software. 


In [101]:
print(ask_question("What is regression analysis"))

a statistical process for estimating the. relationships among variables
